In [ ]:
import subprocess, torch
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run(['pip','install','-q','diffusers','transformers','accelerate'])
import urllib.request
from PIL import Image, ImageOps, ImageDraw, ImageFilter
import numpy as np

BASE = Image.open(urllib.request.urlopen('https://raw.githubusercontent.com/sunraz/shimi-land/main/gallery/lea-chest-fix-17-10/i2g_s101.png')).convert('RGB')
W, H = BASE.size
print('base', BASE.size, flush=True)

from diffusers import StableDiffusionXLInpaintPipeline, StableDiffusionXLImg2ImgPipeline
pipe = StableDiffusionXLInpaintPipeline.from_pretrained('SG161222/RealVisXL_V4.0', torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe.enable_attention_slicing()
print('pipe ready', flush=True)

PROMPT = ('photorealistic photo, the exact same woman as the input, identical face, identical dark hair in a bun, identical pose on the bed, identical background, TOPLESS with bare breasts fully exposed, no clothing, no underwear, no bra, wearing only the same red shorts, hourglass figure with slim waist, flat stomach, very large full heavy natural breasts, enormous breast volume, deep cleavage, rounded teardrop shape with natural weight, realistic skin texture and tone, sharp focus')
NEG = ('bra, bikini top, tank top, sports bra, crop top, clothing on chest, covered breasts, underwear top, ')

mask = Image.new('L', (W, H), 0)
d = ImageDraw.Draw(mask)
cx, cy = W//2, int(H*0.46)
rx, ry = int(W*0.22), int(H*0.12)
d.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=255)
mask = mask.filter(ImageFilter.GaussianBlur(12))
mask.save('/kaggle/working/mask.png')

# שלב 1: הפשטה ובניית נפח
img = pipe(PROMPT, negative_prompt=NEG, image=BASE, mask_image=mask,
           strength=0.80, guidance_scale=7.5, num_inference_steps=40, seed=1111).images[0]
img.save('/kaggle/working/stage1.png')
print('stage1 done', flush=True)
# שלב 2: חידוד ועקביות באותו אזור
img2 = pipe(PROMPT, negative_prompt=NEG, image=img, mask_image=mask,
            strength=0.35, guidance_scale=7.0, num_inference_steps=28, seed=1111+1).images[0]
img2.save('/kaggle/working/l101_top_color.png')
print('stage2 done', flush=True)
ImageOps.autocontrast(img2.convert('L'), cutoff=1).convert('RGB').save('/kaggle/working/l101_top_bw.png')
print('bw done', flush=True)

del pipe
import gc; gc.collect(); torch.cuda.empty_cache()
pipe2 = StableDiffusionXLImg2ImgPipeline.from_pretrained('SG161222/RealVisXL_V4.0', torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe2.enable_attention_slicing()

GPROMPT = ('photorealistic photo, the same woman with identical face and dark hair in a bun, leaning forward toward the camera about 45 degrees on the bed, hands on her thighs, TOPLESS with bare breasts fully exposed, no clothing, no bra, wearing only the same red shorts, hourglass figure with slim waist, her very large heavy natural bare breasts hanging straight down under full gravity, deep natural sag and droop, elongated heavy teardrop shape pulled down by gravity, realistic skin texture, sharp focus')
GNEG = ('floating breasts, perky uplifted breasts, bra, bikini top, clothing on chest, fat, chubby, overweight, belly, small breasts, flat chest, different face, extra breasts, deformed nipples, distorted anatomy, extra limbs, blurry, cartoon, anime, worst quality, low quality, watermark, text')
grav = pipe2(GPROMPT, negative_prompt=GNEG, image=img2, strength=0.55, guidance_scale=7.0,
             num_inference_steps=36, seed=1111+3).images[0]
grav.save('/kaggle/working/l101_grav_color.png')
print('gravity done', flush=True)
ImageOps.autocontrast(grav.convert('L'), cutoff=1).convert('RGB').save('/kaggle/working/l101_grav_bw.png')
print('gravity bw done', flush=True)

b = np.asarray(BASE, dtype=np.float32)
m = np.asarray(mask, dtype=np.float32)/255.0
print('VALIDATION:', flush=True)
for f in ['stage1.png','l101_top_color.png']:
    a = np.asarray(Image.open('/kaggle/working/'+f).convert('RGB'), dtype=np.float32)
    diff = np.abs(a-b).mean(axis=2)
    print(f, 'in-mask', round(float((diff*m).sum()/m.sum()),2),
          'out-mask', round(float((diff*(1-m)).sum()/(1-m).sum()),2), flush=True)
print('ALL DONE', flush=True)